In [ ]:
import pandas as pd
import datetime as dt
import numpy as np
from sqlalchemy import create_engine


In [ ]:
USER = "root"
PASSWORD = "Abhi8383055393"
HOST = "localhost"
PORT = "3306"
DATABASE = "Sales_and_profitability_analysis"

conn_str = f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DATABASE}"

mysql_conn = create_engine(conn_str)

In [ ]:
# orders detail table
orders = pd.read_csv("C:/Users/abhis/OneDrive/Documents/abhishek/Data_Analyst_Abhishek/data_analyst/python_data/practice/Sales & Profitability Analytics/data/raw/orders.csv")

orders['order_date'] = pd.to_datetime(orders['order_date']).dt.date

orders.info()

In [ ]:
# order item table
order_items = pd.read_csv("C:/Users/abhis/OneDrive/Documents/abhishek/Data_Analyst_Abhishek/data_analyst/python_data/practice/Sales & Profitability Analytics/data/raw/order_items.csv")

order_items.info()

In [ ]:
# Product Table
products = pd.read_csv("C:/Users/abhis/OneDrive/Documents/abhishek/Data_Analyst_Abhishek/data_analyst/python_data/practice/Sales & Profitability Analytics/data/raw/products.csv")

products.info()

products[products['category'].isna() == True].count()

products.loc[products['category'].isna(),['subcategory','category']]

products[products['subcategory'] == "Storage"].head(3)

category_map = {
"Printing": "Office Supplies",
"Accessories" : "Electronics",	
"Footwear" : "Clothing",	
"Audio" : "Electronics",	
"Cleaning" : "Home Appliances",	
"Paper" : "Office Supplies",	
"Desks" : "Furniture",	
"Mobiles" : "Electronics",	
"Stationery" : "Office Supplies",	
"Storage" : "Furniture"
}

products['category'] = products['category'].fillna(products['subcategory'].map(category_map))

products.info()

products['launch_date'] = pd.to_datetime(products['launch_date']).dt.date

products.info()


In [ ]:
# region table 
regions = pd.read_csv("C:/Users/abhis/OneDrive/Documents/abhishek/Data_Analyst_Abhishek/data_analyst/python_data/practice/Sales & Profitability Analytics/data/raw/regions.csv")
regions


In [ ]:
# sales channel table 
sales_channels = pd.read_csv("C:/Users/abhis/OneDrive/Documents/abhishek/Data_Analyst_Abhishek/data_analyst/python_data/practice/Sales & Profitability Analytics/data/raw/sales_channels.csv")
sales_channels

In [ ]:
# returns 
returns = pd.read_csv("C:/Users/abhis/OneDrive/Documents/abhishek/Data_Analyst_Abhishek/data_analyst/python_data/practice/Sales & Profitability Analytics/data/raw/returns.csv")
returns['return_date'] = pd.to_datetime(returns['return_date']).dt.date
returns.info()

In [ ]:
# target 
targets = pd.read_csv("C:/Users/abhis/OneDrive/Documents/abhishek/Data_Analyst_Abhishek/data_analyst/python_data/practice/Sales & Profitability Analytics/data/raw/targets.csv")
targets.info()

In [21]:
# export unclean data into mysql
order_items.to_sql("order_items",mysql_conn,if_exists='replace',index=False)
products.to_sql("products",mysql_conn,if_exists='replace',index=False)
returns.to_sql("returns",mysql_conn,if_exists='replace',index=False)
targets.to_sql("targets",mysql_conn,if_exists='replace',index=False)
regions.to_sql("regions",mysql_conn,if_exists='replace',index=False)
sales_channels.to_sql("sales_channels",mysql_conn,if_exists='replace',index=False)


4

In [22]:
order_items.columns

Index(['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price',
       'discount_pct', 'unit_cost'],
      dtype='str')

In [23]:
products.columns

Index(['product_id', 'product_name', 'category', 'subcategory', 'brand',
       'unit_cost', 'standard_price', 'supplier_id', 'launch_date'],
      dtype='str')

In [24]:
# merging the tables 
full_order_details = pd.merge(order_items,products,how='right',on='product_id')

In [27]:
main_fact_table = pd.merge(
    full_order_details,
    orders,
    how='right',
    on='order_id'
)

In [28]:
main_fact_table.head()

,order_item_id,order_id,product_id,quantity,unit_price,discount_pct,unit_cost_x,product_name,category,subcategory,...,unit_cost_y,standard_price,supplier_id,launch_date,order_date,customer_id,region_id,channel_id,payment_method,order_status
0,OI00000001,ORD0000001,P00969,3.0,3001.37,0.1,2504.00,Office Supplies Product 0969,office supplies,Stationery,...,2504.00,3062.68,S0148,2023-11-30,2025-12-01,C00024,R02,CH01,COD,Completed
1,OI00000003,ORD0000001,P00979,1.0,1501.11,0.3,1117.15,Office Supplies Product 0979,Office Supplies,Printing,...,1117.15,1504.38,S0163,2025-11-30,2025-12-01,C00024,R02,CH01,COD,Completed
2,OI00000002,ORD0000001,P01211,2.0,542.67,0.0,452.92,Home Appliances Product 1211,Home Appliances,Kitchen,...,452.92,591.27,S0147,2024-12-07,2025-12-01,C00024,R02,CH01,COD,Completed
3,OI00000005,ORD0000002,P00823,3.0,975.95,0.3,655.17,Furniture Product 0823,Furniture,Home Office,...,655.17,1026.61,S0189,2022-02-26,2025-01-29,C06641,R07,CH01,Net Banking,Completed
4,OI00000004,ORD0000002,P01465,1.0,1124.00,0.2,975.46,Clothing Product 1465,Clothing,Men,...,975.46,1123.77,S0056,2021-09-08,2025-01-29,C06641,R07,CH01,Net Banking,Completed


In [35]:
main_fact_table.isna().value_counts()

order_item_id  order_id  product_id  quantity  unit_price  discount_pct  unit_cost_x  product_name  category  subcategory  brand  unit_cost_y  standard_price  supplier_id  launch_date  order_date  customer_id  region_id  channel_id  payment_method  order_status
False          False     False       False     False       False         False        False         False     False        False  False        False           False        False        False       False        False      False       False           False           257406
True           False     True        True      True        True          True         True          True      True         True   True         True            True         True         False       False        False      False       False           False               11
Name: count, dtype: int64

In [36]:
main_fact_table['order_item_id'].isna().value_counts()

order_item_id
False    257406
True         11
Name: count, dtype: int64

In [ ]:
null_order_item_rows = main_fact_table[main_fact_table['order_item_id'].isna() == True]
null_order_item_rows

In [40]:
main_fact_table.isna().value_counts()

order_item_id  order_id  product_id  quantity  unit_price  discount_pct  unit_cost_x  product_name  category  subcategory  brand  unit_cost_y  standard_price  supplier_id  launch_date  order_date  customer_id  region_id  channel_id  payment_method  order_status
False          False     False       False     False       False         False        False         False     False        False  False        False           False        False        False       False        False      False       False           False           257406
True           False     True        True      True        True          True         True          True      True         True   True         True            True         True         False       False        False      False       False           False               11
Name: count, dtype: int64

In [41]:
main_fact_table['product_id'].isna().value_counts()

product_id
False    257406
True         11
Name: count, dtype: int64

In [ ]:
main_fact_table[main_fact_table['product_id'].isna() == True]

In [44]:
main_fact_table = main_fact_table.dropna(subset=['order_item_id'])

In [ ]:
main_fact_table.isna().value_counts()


In [47]:
order_items.shape

(256156, 7)